In [1]:
import torch
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# ==========================================
# 1. PYTORCH & GPU VERIFICATION
# ==========================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🤖 PyTorch is currently using: {device.upper()}")

if torch.cuda.is_available():
    print(f"🔥 GPU name: {torch.cuda.get_device_name(0)}")

🤖 PyTorch is currently using: CUDA
🔥 GPU name: NVIDIA RTX A5000 Laptop GPU


In [2]:
# ==========================================
# 2. EMBEDDINGS (PyTorch on GPU)
# ==========================================
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True}
)


# ==========================================
# 3. CREATE A MINI KNOWLEDGE BASE (RAG)
# ==========================================
facts = [
    "The current system is a stable MLOps stack.",
    "It runs on WSL2 with MicroK8s and Charmed Kubeflow 1.10.",
    "MLflow is integrated.",
    "GPU acceleration works in notebooks.",
    "GPU acceleration works in pipeline pods via Kyverno.",
    "Ollama now runs in Ubuntu and is reachable through a Kubernetes Service."
]

knowledge_base = Chroma.from_texts(facts, embeddings)
retriever = knowledge_base.as_retriever(search_kwargs={"k": 2})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
# ==========================================
# 4. OLLAMA & LANGCHAIN INTEGRATION
# ==========================================
llm = ChatOllama(
    model="gemma2:latest",
    temperature=0,
    base_url="http://ollama.kubeflow.svc.cluster.local:11434"
)

prompt_template = ChatPromptTemplate.from_template("""
Answer the question only based on the following context.
If you do not know the answer, say: "I don't know."

Context:
{context}

Question:
{question}

Answer:
""")

In [4]:
# ==========================================
# 5. RUN THE AGENT QUERY
# ==========================================
question = "What does the stable MLOps stack run on?"

relevant_documents = retriever.invoke(question)
context = "\n".join(doc.page_content for doc in relevant_documents)

prompt = prompt_template.format(context=context, question=question)
answer = llm.invoke(prompt)

print(f"\nQuestion: {question}")
print(f"\nContext:\n{context}")
print(f"\nAnswer from local LLM:\n{answer.content}")


Question: What does the stable MLOps stack run on?

Context:
The current system is a stable MLOps stack.
MLflow is integrated.

Answer from local LLM:
I don't know. 



In [5]:
retriever = knowledge_base.as_retriever(search_kwargs={"k": 4})

Was passiert hier genau?PyTorch prüft Ihre CUDA-Treiber und lädt ein HuggingFace-Modell direkt in den VRAM Ihrer GPU.LangChain wandelt Ihre Textfakten mithilfe dieses GPU-Modells blitzschnell in mathematische Vektoren um.Wenn Sie eine Frage stellen, sucht das System den passenden Fakt heraus und sendet ihn zusammen mit der Frage an Ollama, welches die finale, menschliche Antwort generiert.